# Assignment 3 - Part II

In this assignment, I will explore the use of Large Language Models (LLMs) for text classification by predicting the genre of song lyrics. The LLM accessed for this assignment is Ollama. The goal is to evaluate how well Ollama can classify lyrics into the defined genres using different prompting strategies.

Two approaches are implemented: **zero-shot prompting**, where the model is given only instructions without examples, and **few-shot prompting**, where the model is provided with a small number of labeled examples to guide its predictions. The performance of both approaches is evaluated using standard classification metrics: _precision_, _recall_, and _F1-score_. Additionally, results are analyzed per genre to identify strengths and weaknesses of the model.

## Downloads and Imports

Importing __Ollama__ as the Large Language Model, __Pandas__ to handle the dataset, and __sklearn__ to evaluate performance. __Re__ is imported for using search patterns. 

> If Ollama is not already downloaded on your device, you should install it using this link: https://ollama.com/download. 

In [1]:
import pandas as pd
import ollama
from sklearn.metrics import classification_report
import re

## Import the Data

Next, the train and test datasets are loaded. The separator is set to __\t__ because the file is tab-separated. I inspect the data to ensure it loaded correctly.

In [2]:
train_df = pd.read_csv("genreLyrics_train.csv", sep="\t")
test_df = pd.read_csv("genreLyrics_test.csv", sep="\t")

print(train_df.head())
print(test_df.head())

   Unnamed: 0       genre                                             lyrics
0      263935        Rock  I knew a man, called him Sandy Cane\nFew folks...
1       64235     Country  (Gary Harrison - Kent Robbins - Gene Miller)\n...
2      197695        Rock  You think its right when you see my reaction\n...
3      274428  Electronic  Darkness, you are gentle when you kiss me\nYou...
4      116870        Rock  Cold fire, you've got everything but cold fire...
   Unnamed: 0       genre                                             lyrics
0      253513     Hip-Hop  State Property Music\nuh, holla, uh...... yeah...
1      218012     Country  Well I really had a ball last night\nI held al...
2       91614        Rock  [lyrics:Stefan Ruiters]\nsolar child is burnin...
3      320535  Electronic  La milonga.\nLadies and gentlemen\nA man.\nMil...
4      248213         Pop  Anybody ever tell you that you're not whole\nA...


## Functions

### Clean Model Output

LLMs often return extra text. This function extracts only the genre label from the model’s response to ensure consistent evaluation.

In [3]:
def extract_label(response, valid_labels):
    response = response.lower()
    
    for label in valid_labels:
        if label.lower() in response:
            return label
    
    return "unknown"

### Labels 

By extracting the labels in the dataset, I can use them as valid classification labels for the model.

In [4]:
labels = sorted(train_df["genre"].unique())
print(labels)

['Country', 'Electronic', 'Folk', 'Hip-Hop', 'Indie', 'Jazz', 'Metal', 'Other', 'Pop', 'R&B', 'Rock']


## Zero Shot Prompting

This function performs __zero-shot prompting__, where the model is asked to classify lyrics without seeing any examples.

In [5]:
def zero_shot_predict(lyrics):
    prompt = f"""
    Classify the genre of the following song lyrics.
    Choose ONLY one from this list: {labels}.
    
    Lyrics:
    {lyrics}
    
    Answer with only the genre name.
    """

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )

    return response['message']['content']

## Few Shot Prompting 

This function builds a __few-shot prompt__ by adding example lyrics and their genres. This helps the model learn patterns before making a prediction.

In [6]:
def build_few_shot_prompt(lyrics, k=3):
    examples = train_df.sample(k)
    
    example_text = ""
    for _, row in examples.iterrows():
        example_text += f"""
        Lyrics: {row['lyrics']}
        Genre: {row['genre']}
        """

    prompt = f"""
    Classify song lyrics into genres.

    Possible genres: {labels}

    Examples:
    {example_text}

    Now classify:

    Lyrics:
    {lyrics}

    Please answer with only the genre name.
    """
    
    return prompt

This function uses the few-shot prompt to classify lyrics. 

In [7]:
def few_shot_predict(lyrics):
    prompt = build_few_shot_prompt(lyrics)

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )

    return response['message']['content']

## Run Predicitions on a Subset

Here, I will run predictions on a small subset of the test data. Results are stored for evaluation.

In [ ]:
subset = test_df.sample(100, random_state=42)

y_true = []
y_pred_zero = []
y_pred_few = []

for _, row in subset.iterrows():
    lyrics = row["lyrics"]
    true_label = row["genre"]

    # Zero-shot
    z_pred_raw = zero_shot_predict(lyrics)
    z_pred = extract_label(z_pred_raw, labels)

    # Few-shot
    f_pred_raw = few_shot_predict(lyrics)
    f_pred = extract_label(f_pred_raw, labels)

    y_true.append(true_label)
    y_pred_zero.append(z_pred)
    y_pred_few.append(f_pred)

## Evaluation

I compute Precision, Recall, and F1-score to evaluate how well each prompting strategy performs.

In [ ]:
print("ZERO-SHOT PERFORMANCE")
print(classification_report(y_true, y_pred_zero))

print("FEW-SHOT PERFORMANCE")
print(classification_report(y_true, y_pred_few))

## Genre-Wise Performance

This step analyzes performance per genre to see which genres are easier or harder to classify.

In [ ]:
report_zero = classification_report(y_true, y_pred_zero, output_dict=True)
report_few = classification_report(y_true, y_pred_few, output_dict=True)

pd.DataFrame(report_zero).transpose()
pd.DataFrame(report_few).transpose()

## Inspect Outputs

In [ ]:
for i in range(5):
    print("RAW:", y_pred_zero[i])